# 🎯 Model Evaluation Notebook
Load any checkpoint and evaluate it thoroughly.

In [ ]:
import sys,os
sys.path.insert(0,os.path.abspath('..'))
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from model.environment import CartPoleEnv
from model.dqn_agent import DQNAgent
print('Ready')

In [ ]:
# Load best model
agent = DQNAgent.load_checkpoint('../checkpoints/best_model.pkl')
print(agent.summary())

In [ ]:
# 30-episode greedy evaluation
env=CartPoleEnv(seed=5000); rewards=[]
for ep in range(30):
    obs=env.reset(seed=5000+ep); tot=0.
    for _ in range(500):
        a=agent.select_action_greedy(obs); obs,r,t,tr,_=env.step(a); tot+=r
        if t or tr: break
    rewards.append(tot)
print(f'Mean={np.mean(rewards):.1f}  Std={np.std(rewards):.1f}  Max={max(rewards):.0f}  Solve={np.mean(np.array(rewards)>=475)*100:.0f}%')

In [ ]:
# Q-value probe
probe=np.array([[0,0,0,0],[0,0,.1,0],[0,0,-.1,0],[1,.5,.05,.1]],dtype=np.float32)
labels=['Balanced','Lean Right','Lean Left','Drifting']
print(f'{"State":<14} {"Q(Left)":>10} {"Q(Right)":>10} {"Action":>10}')
print('-'*48)
for lbl,s in zip(labels,probe):
    q=agent.q_net.predict(s.reshape(1,-1))[0]
    print(f'{lbl:<14} {q[0]:>10.2f} {q[1]:>10.2f} {["LEFT","RIGHT"][np.argmax(q)]:>10}')

In [ ]:
# ASCII render of one episode
env2=CartPoleEnv(seed=42); obs=env2.reset(seed=42); total=0.
print('Episode render:')
for step in range(50):
    a=agent.select_action_greedy(obs); obs,r,t,tr,_=env2.step(a); total+=r
    if step%5==0: print(f'  {env2.render()}')
    if t or tr: break
print(f'Total reward: {total:.0f}  Steps: {env2.step_count}')

In [ ]:
# Compare checkpoints
checkpoints={'Pretrained':'../checkpoints/pretrained_model.pkl',
             'Ep 100':'../checkpoints/checkpoint_ep100.pkl',
             'Ep 200':'../checkpoints/checkpoint_ep200.pkl',
             'Best':'../checkpoints/best_model.pkl'}
env3=CartPoleEnv(seed=7777)
results={}
for name,path in checkpoints.items():
    if not os.path.exists(path): continue
    ag=DQNAgent.load_checkpoint(path); evr=[]
    for ei in range(10):
        obs=env3.reset(seed=7777+ei); tot=0.
        for _ in range(500):
            a=ag.select_action_greedy(obs); obs,r,t,tr,_=env3.step(a); tot+=r
            if t or tr: break
        evr.append(tot)
    results[name]=evr
    print(f'{name:<12}: mean={np.mean(evr):.1f}  max={max(evr):.0f}')